In [1]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim

from torchvision import transforms
from torch.utils.data import Dataset
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from PIL import Image
from torchsummary import summary

In [2]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
print(torch.cuda.is_available())

True


In [4]:
print(torch.__version__)

2.9.1+cu128


As I am using images I have to load them as datasets

It seems that child class has to define its own implementation of the __len__ and __getitem__ methods

In [5]:
class HairDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.dataset = ImageFolder(root=root_dir, transform=transform)
        self.samples = self.dataset.samples  #(img_path, class_idx)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

I am following the example in the workshop notebook to apply transformations (part of data augmentation)

The mean and std seem to be default values

In [6]:
input_size = 224

#ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

#transforms - just resize and normalize
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(input_size, scale=(0.9, 1.0)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [7]:
#define transformations
train_dataset = HairDataset(
    root_dir="../data/train/",
    transform=train_transforms
)

val_dataset = HairDataset(
    root_dir="../data/test/",
    transform=val_transforms
)

In [8]:
#create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

Here is an attempt to create a classifier for the available data adding custom layers

In [9]:
class HairClassifierMobileNet(nn.Module):
    def __init__(self, num_classes=2):
        super(HairClassifierMobileNet, self).__init__()

        # Load pre-trained MobileNetV2
        self.base_model = models.mobilenet_v2(weights="IMAGENET1K_V1")

        # Freeze base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Remove original classifier
        self.base_model.classifier = nn.Identity()

        # Add custom layers
        self.global_avg_pooling = nn.AdaptiveAvgPool2d((1, 1))
        #match size of the input tensor's last dimension
        self.output_layer = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.base_model.features(x)
        x = self.global_avg_pooling(x)
        x = torch.flatten(x, 1)
        x = self.output_layer(x)
        return x

Getting number of classes in one of the datasets (both have the same number of classes)

In [10]:
classes = len(train_dataset.dataset.classes)
classes

2

Just forcing use of cuda if available

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Intantiate a model with desired number of classes and using cuda

In [12]:
model = HairClassifierMobileNet(num_classes=classes)
model.to(device)

HairClassifierMobileNet(
  (base_model): MobileNetV2(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU6(inplace=True)
          )
          (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (2): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 96,

Creating an optimizer so the model can learn in this case using a learning rate value of 0.01 is used and criterion CrossEntropyLoss as the task is a classification problem

In [13]:
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

Training and validation processes

In [14]:
# Training loop
num_epochs = 10

for epoch in range(num_epochs):
    # Training phase
    model.train()  # Set the model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Iterate over the training data
    for inputs, labels in train_loader:
        # Move data to the specified device (GPU or CPU)
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients to prevent accumulation
        optimizer.zero_grad()
        # Forward pass
        outputs = model(inputs)
        # Calculate the loss
        loss = criterion(outputs, labels)
        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate training loss
        running_loss += loss.item()
        # Get predictions
        _, predicted = torch.max(outputs.data, 1)
        # Update total and correct predictions
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Calculate average training loss and accuracy
    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # Validation phase
    model.eval()  # Set the model to evaluation mode
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    # Disable gradient calculation for validation
    with torch.no_grad():
        # Iterate over the validation data
        for inputs, labels in val_loader:
            # Move data to the specified device (GPU or CPU)
            inputs, labels = inputs.to(device), labels.to(device)
            # Forward pass
            outputs = model(inputs)
            # Calculate the loss
            loss = criterion(outputs, labels)

            # Accumulate validation loss
            val_loss += loss.item()
            # Get predictions
            _, predicted = torch.max(outputs.data, 1)
            # Update total and correct predictions
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    # Calculate average validation loss and accuracy
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    # Print epoch results
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

Epoch 1/10
  Train Loss: 0.8874, Train Acc: 0.7700
  Val Loss: 0.0591, Val Acc: 0.9751
Epoch 2/10
  Train Loss: 0.1116, Train Acc: 0.9537
  Val Loss: 0.0403, Val Acc: 0.9851
Epoch 3/10
  Train Loss: 0.0911, Train Acc: 0.9650
  Val Loss: 0.2094, Val Acc: 0.9154
Epoch 4/10
  Train Loss: 0.1668, Train Acc: 0.9500
  Val Loss: 0.0412, Val Acc: 0.9900
Epoch 5/10
  Train Loss: 0.0663, Train Acc: 0.9750
  Val Loss: 0.0473, Val Acc: 0.9851
Epoch 6/10
  Train Loss: 0.0485, Train Acc: 0.9825
  Val Loss: 0.0445, Val Acc: 0.9801
Epoch 7/10
  Train Loss: 0.0623, Train Acc: 0.9800
  Val Loss: 0.0327, Val Acc: 0.9900
Epoch 8/10
  Train Loss: 0.0284, Train Acc: 0.9888
  Val Loss: 0.0354, Val Acc: 0.9851
Epoch 9/10
  Train Loss: 0.0400, Train Acc: 0.9825
  Val Loss: 0.1102, Val Acc: 0.9453
Epoch 10/10
  Train Loss: 0.0907, Train Acc: 0.9575
  Val Loss: 0.0469, Val Acc: 0.9801


Using functions to do previous steps

In [15]:
def make_model(learning_rate=0.01):
    model = HairClassifierMobileNet(num_classes=2)
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    return model, optimizer

In [16]:
criterion = nn.CrossEntropyLoss()
num_epochs = 10

In [17]:
def train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device, version):
    best_val_accuracy = 0.0  # Initialize variable to track the best validation accuracy

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total

        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc
            checkpoint_path = f"hair_pth_{version}_{epoch+1:02d}_{val_acc:.4f}.pth"
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Checkpoint saved: {checkpoint_path}")

Optimizing learning rate and saving best max results, best result is saved each time val__acc is higher than previous best val_acc

In [18]:
version = "v1_lr"
for lr in [0.001, 0.1]: #0.01, 0.1]:
    print("learning rate =", lr)
    model, optimizer = make_model(lr)
    train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device, version)

learning rate = 0.001
Epoch 1/10
  Train Loss: 0.3866, Train Acc: 0.8588
  Val Loss: 0.2679, Val Acc: 0.9254
Checkpoint saved: hair_pth_v1_lr_01_0.9254.pth
Epoch 2/10
  Train Loss: 0.2082, Train Acc: 0.9425
  Val Loss: 0.1897, Val Acc: 0.9154
Epoch 3/10
  Train Loss: 0.1432, Train Acc: 0.9675
  Val Loss: 0.1161, Val Acc: 0.9900
Checkpoint saved: hair_pth_v1_lr_03_0.9900.pth
Epoch 4/10
  Train Loss: 0.1299, Train Acc: 0.9637
  Val Loss: 0.1030, Val Acc: 0.9900
Epoch 5/10
  Train Loss: 0.1218, Train Acc: 0.9587
  Val Loss: 0.0961, Val Acc: 0.9751
Epoch 6/10
  Train Loss: 0.1135, Train Acc: 0.9688
  Val Loss: 0.0920, Val Acc: 0.9652
Epoch 7/10
  Train Loss: 0.1085, Train Acc: 0.9575
  Val Loss: 0.0747, Val Acc: 0.9900
Epoch 8/10
  Train Loss: 0.0825, Train Acc: 0.9788
  Val Loss: 0.0742, Val Acc: 0.9900
Epoch 9/10
  Train Loss: 0.0844, Train Acc: 0.9788
  Val Loss: 0.0787, Val Acc: 0.9851
Epoch 10/10
  Train Loss: 0.0832, Train Acc: 0.9712
  Val Loss: 0.0634, Val Acc: 0.9851
learning rate

Using droprate, inner_size

In [19]:
class HairClassifierMobileNet(nn.Module):
    def __init__(self, size_inner=100, droprate=0.2, num_classes=classes):
        super(HairClassifierMobileNet, self).__init__()

        # Load pre-trained MobileNetV2
        self.base_model = models.mobilenet_v2(weights="IMAGENET1K_V1")

        # Freeze base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Remove original classifier
        self.base_model.classifier = nn.Identity()

        # Add custom layers
        self.global_avg_pooling = nn.AdaptiveAvgPool2d((1, 1))

        self.inner = nn.Linear(1280, size_inner)  # New inner layer
        self.relu = nn.ReLU() # activation function
        self.dropout = nn.Dropout(droprate)  # Add dropout
        self.output_layer = nn.Linear(size_inner, num_classes)

    def forward(self, x):
        x = self.base_model.features(x)
        x = self.global_avg_pooling(x)
        x = torch.flatten(x, 1)
        x = self.inner(x)
        x = self.relu(x)
        x = self.dropout(x)  # Apply dropout
        x = self.output_layer(x)
        return x

Adding new steps to a function to build a model

In [20]:
def make_model_dr_sz(learning_rate=0.01, size_inner=100, droprate=0.2):
    model = HairClassifierMobileNet(
        num_classes=classes,
        size_inner=size_inner,
        droprate=droprate,
    )
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    return model, optimizer

In [21]:
model, optimizer = make_model_dr_sz(
    learning_rate=0.001,
    size_inner=100,
    droprate=0.2,
)

criterion = nn.CrossEntropyLoss()
num_epochs = 50
version = "v2_dr_sz"

train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device, version)

Epoch 1/50
  Train Loss: 0.4804, Train Acc: 0.7800
  Val Loss: 0.2012, Val Acc: 0.9701
Checkpoint saved: hair_pth_v2_dr_sz_01_0.9701.pth
Epoch 2/50
  Train Loss: 0.1554, Train Acc: 0.9613
  Val Loss: 0.1169, Val Acc: 0.9552
Epoch 3/50
  Train Loss: 0.1226, Train Acc: 0.9587
  Val Loss: 0.0817, Val Acc: 0.9701
Epoch 4/50
  Train Loss: 0.1137, Train Acc: 0.9663
  Val Loss: 0.0724, Val Acc: 0.9701
Epoch 5/50
  Train Loss: 0.1557, Train Acc: 0.9450
  Val Loss: 0.0736, Val Acc: 0.9751
Checkpoint saved: hair_pth_v2_dr_sz_05_0.9751.pth
Epoch 6/50
  Train Loss: 0.0819, Train Acc: 0.9675
  Val Loss: 0.0711, Val Acc: 0.9751
Epoch 7/50
  Train Loss: 0.0818, Train Acc: 0.9750
  Val Loss: 0.0487, Val Acc: 0.9801
Checkpoint saved: hair_pth_v2_dr_sz_07_0.9801.pth
Epoch 8/50
  Train Loss: 0.0573, Train Acc: 0.9812
  Val Loss: 0.0654, Val Acc: 0.9701
Epoch 9/50
  Train Loss: 0.0764, Train Acc: 0.9725
  Val Loss: 0.0489, Val Acc: 0.9751
Epoch 10/50
  Train Loss: 0.0747, Train Acc: 0.9738
  Val Loss: 0.0

In [22]:
summary(model, input_size=(3, 224, 224))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 112, 112]             864
       BatchNorm2d-2         [-1, 32, 112, 112]              64
             ReLU6-3         [-1, 32, 112, 112]               0
            Conv2d-4         [-1, 32, 112, 112]             288
       BatchNorm2d-5         [-1, 32, 112, 112]              64
             ReLU6-6         [-1, 32, 112, 112]               0
            Conv2d-7         [-1, 16, 112, 112]             512
       BatchNorm2d-8         [-1, 16, 112, 112]              32
  InvertedResidual-9         [-1, 16, 112, 112]               0
           Conv2d-10         [-1, 96, 112, 112]           1,536
      BatchNorm2d-11         [-1, 96, 112, 112]             192
            ReLU6-12         [-1, 96, 112, 112]               0
           Conv2d-13           [-1, 96, 56, 56]             864
      BatchNorm2d-14           [-1, 96,

There are 2M+ non-trainable parameters

Load a saved model

In [23]:
model_path = "./hair_pth_v2_dr_sz_42_0.9900.pth"

In [24]:
#def __init__(self, size_inner=100, droprate=0.2, num_classes=classes):
model_load = HairClassifierMobileNet(size_inner=100, droprate=0.2, num_classes=classes)
model_load.load_state_dict(torch.load(model_path))
model_load.to(device)
model_load.eval()

HairClassifierMobileNet(
  (base_model): MobileNetV2(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU6(inplace=True)
          )
          (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (2): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 96,

In [25]:
img = Image.open("../data/test/curly/61aPFVrm42L._SL1352_.jpg")
x = val_transforms(img)
batch_t = torch.unsqueeze(x, 0).to(device)

with torch.no_grad():
    output = model(batch_t)

In [26]:
classes = train_dataset.dataset.classes
classes

['curly', 'straight']

In [27]:
class_probs = dict(zip(classes, output[0].to("cpu")))
class_probs

{'curly': tensor(4.6576), 'straight': tensor(-4.8197)}

In [28]:
prob_class = max(class_probs, key=class_probs.get)
print(f"Probable class: {prob_class} with probability: {class_probs.get(prob_class)}")

Probable class: curly with probability: 4.6575927734375
